# ToolForge: Fine-Tuning Qwen2.5 for Reliable Tool Calling

This notebook fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with QLoRA on the `NousResearch/hermes-function-calling-v1` dataset, then evaluates the base model and fine-tuned adapter on structured tool-calling metrics.


## 1. Install Dependencies

!pip install -q -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub gradio pandas scikit-learn

## 2. Imports and Configuration

In [ ]:
import ast
import gc
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "NousResearch/hermes-function-calling-v1"

TRAIN_SAMPLES = 2500
TEST_SAMPLES = 200
MAX_SEQ_LENGTH = 2048

PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3

PROJECT_DIR = Path("/kaggle/working/toolforge")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
ADAPTER_DIR = PROJECT_DIR / "toolforge-qwen2.5-1.5b-lora"

for path in [PROJECT_DIR, DATA_DIR, RESULTS_DIR, ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Optional Hugging Face Login

In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face Hub.")

## 4. Load and Inspect the Dataset

In [ ]:
raw = load_dataset(DATASET_ID)
print(raw)

if isinstance(raw, DatasetDict):
    split_name = "train" if "train" in raw else list(raw.keys())[0]
    raw_dataset = raw[split_name]
else:
    raw_dataset = raw

print("Using split size:", len(raw_dataset))
print("Columns:", raw_dataset.column_names)
print(json.dumps(raw_dataset[0], indent=2)[:3000])

## 5. Normalize Conversations into Training Examples

Each training row becomes a single text string generated with Qwen's chat template:

- `system`: instruction and tool context.
- `user`: user request.
- `assistant`: expected tool-call response.

The evaluation code also stores parsed expected tool calls where possible.


In [ ]:
def maybe_parse_json(value: Any) -> Any:
    if isinstance(value, (dict, list)):
        return value
    if not isinstance(value, str):
        return value
    text = value.strip()
    if not text:
        return value
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return value


def stringify_tool_schema(tools: Any) -> str:
    tools = maybe_parse_json(tools)
    if tools is None or tools == "":
        return "[]"
    if isinstance(tools, str):
        return tools
    return json.dumps(tools, ensure_ascii=False, indent=2)


def role_of(message: Dict[str, Any]) -> str:
    return str(message.get("role") or message.get("from") or message.get("speaker") or "").lower()


def content_of(message: Dict[str, Any]) -> str:
    value = message.get("content")
    if value is None:
        value = message.get("value")
    if value is None:
        value = message.get("text")
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return "" if value is None else str(value)


def extract_messages(row: Dict[str, Any]) -> Optional[List[Dict[str, str]]]:
    candidate_keys = ["messages", "conversations", "conversation", "chat"]
    for key in candidate_keys:
        if key in row:
            parsed = maybe_parse_json(row[key])
            if isinstance(parsed, list):
                messages = []
                for item in parsed:
                    if isinstance(item, dict):
                        messages.append({"role": role_of(item), "content": content_of(item)})
                if messages:
                    return messages

    system = row.get("system") or row.get("system_prompt") or row.get("instruction")
    user = row.get("user") or row.get("prompt") or row.get("input") or row.get("question")
    assistant = row.get("assistant") or row.get("output") or row.get("response") or row.get("answer")

    if user is not None and assistant is not None:
        messages = []
        if system:
            messages.append({"role": "system", "content": str(system)})
        messages.append({"role": "user", "content": str(user)})
        messages.append({"role": "assistant", "content": str(assistant)})
        return messages

    return None


def split_messages(messages: List[Dict[str, str]], tools: Any = None) -> Optional[Dict[str, Any]]:
    system_parts = []
    user_parts = []
    assistant_parts = []

    for msg in messages:
        role = msg["role"]
        content = msg["content"].strip()
        if not content:
            continue
        if role in ["system"]:
            system_parts.append(content)
        elif role in ["user", "human"]:
            user_parts.append(content)
        elif role in ["assistant", "gpt", "model"]:
            assistant_parts.append(content)

    if not user_parts or not assistant_parts:
        return None

    tool_schema = stringify_tool_schema(tools)
    default_system = (
        "You are ToolForge, a helpful assistant that returns structured tool calls. "
        "When a tool is needed, respond only with one or more <tool_call> blocks. "
        "Each block must contain valid JSON with keys 'name' and 'arguments'. "
        "Do not call tools that are not available."
    )
    system_text = "\n\n".join(system_parts).strip() or default_system

    if tool_schema and tool_schema != "[]" and "Available tools" not in system_text:
        system_text = f"{system_text}\n\nAvailable tools:\n{tool_schema}"

    return {
        "system": system_text,
        "user": "\n\n".join(user_parts).strip(),
        "assistant": "\n\n".join(assistant_parts).strip(),
        "tools": tool_schema,
    }


def row_to_example(row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    tools = row.get("tools") or row.get("available_tools") or row.get("functions") or row.get("apis")
    messages = extract_messages(row)
    if messages is None:
        return None
    return split_messages(messages, tools=tools)


examples = []
for row in raw_dataset:
    example = row_to_example(row)
    if example is not None:
        examples.append(example)

print("Normalized examples:", len(examples))
print(json.dumps(examples[0], indent=2)[:3000])

## 6. Load Tokenizer and Build Chat-Template Text

The model should learn the exact assistant response. We use the tokenizer chat template so training and inference share the same conversation format.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def build_messages(example: Dict[str, Any], include_assistant: bool = True) -> List[Dict[str, str]]:
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
    ]
    if include_assistant:
        messages.append({"role": "assistant", "content": example["assistant"]})
    return messages


def format_for_training(example: Dict[str, Any]) -> str:
    return tokenizer.apply_chat_template(
        build_messages(example, include_assistant=True),
        tokenize=False,
        add_generation_prompt=False,
    )


def format_for_generation(example: Dict[str, Any]) -> str:
    return tokenizer.apply_chat_template(
        build_messages(example, include_assistant=False),
        tokenize=False,
        add_generation_prompt=True,
    )


for ex in examples:
    ex["text"] = format_for_training(ex)
    ex["prompt"] = format_for_generation(ex)
    ex["n_tokens"] = len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"])

filtered = [ex for ex in examples if ex["n_tokens"] <= MAX_SEQ_LENGTH]
print("Examples <= max length:", len(filtered))
print("Token length summary:")
print(pd.Series([ex["n_tokens"] for ex in filtered]).describe())

## 7. Create Train and Test Splits

In [ ]:
random.shuffle(filtered)

needed = TRAIN_SAMPLES + TEST_SAMPLES
if len(filtered) < needed:
    print(f"Warning: requested {needed} examples but only found {len(filtered)} after filtering.")

test_examples = filtered[: min(TEST_SAMPLES, len(filtered))]
train_examples = filtered[min(TEST_SAMPLES, len(filtered)) : min(needed, len(filtered))]

train_ds = Dataset.from_list(train_examples)
test_ds = Dataset.from_list(test_examples)

splits = DatasetDict({"train": train_ds, "test": test_ds})
splits.save_to_disk(str(DATA_DIR / "processed_splits"))

print(splits)
print("Train sample text preview:\n")
print(train_ds[0]["text"][:3000])

## 8. Tool-Call Parsing and Metrics

The metrics are designed for portfolio reporting:

- `format_valid`: At least one valid JSON tool call can be parsed, or both prediction and reference contain no call.
- `function_accuracy`: Exact function-name match across parsed calls.
- `argument_f1`: Token-level F1 over flattened argument key-value pairs.
- `hallucination_rate`: The prediction calls a tool not present in the provided tool list.


In [ ]:
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL | re.IGNORECASE)


def parse_json_object(text: str) -> Optional[Any]:
    text = text.strip()
    candidates = [text]

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace >= 0 and last_brace > first_brace:
        candidates.append(text[first_brace : last_brace + 1])

    first_bracket = text.find("[")
    last_bracket = text.rfind("]")
    if first_bracket >= 0 and last_bracket > first_bracket:
        candidates.append(text[first_bracket : last_bracket + 1])

    for candidate in candidates:
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(candidate)
            except Exception:
                pass
    return None


def normalize_call(obj: Any) -> Optional[Dict[str, Any]]:
    if not isinstance(obj, dict):
        return None

    name = obj.get("name") or obj.get("function") or obj.get("tool") or obj.get("tool_name")
    arguments = obj.get("arguments")
    if arguments is None:
        arguments = obj.get("parameters") or obj.get("args") or {}

    if isinstance(name, dict):
        arguments = name.get("arguments") or arguments
        name = name.get("name")

    if isinstance(arguments, str):
        parsed_args = parse_json_object(arguments)
        if parsed_args is not None:
            arguments = parsed_args

    if not name:
        return None

    return {"name": str(name), "arguments": arguments if arguments is not None else {}}


def parse_tool_calls(text: str) -> Tuple[List[Dict[str, Any]], bool]:
    if not isinstance(text, str):
        return [], False

    chunks = TOOL_CALL_RE.findall(text)
    if not chunks:
        parsed = parse_json_object(text)
        chunks = []
        if isinstance(parsed, list):
            calls = [normalize_call(item) for item in parsed]
            calls = [call for call in calls if call is not None]
            return calls, bool(calls)
        call = normalize_call(parsed)
        return ([call], True) if call else ([], False)

    calls = []
    all_valid = True
    for chunk in chunks:
        parsed = parse_json_object(chunk)
        call = normalize_call(parsed)
        if call is None:
            all_valid = False
        else:
            calls.append(call)
    return calls, all_valid and bool(calls)


def flatten_args(value: Any, prefix: str = "") -> List[str]:
    items = []
    if isinstance(value, dict):
        for key in sorted(value.keys()):
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            items.extend(flatten_args(value[key], next_prefix))
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            items.extend(flatten_args(item, f"{prefix}[{idx}]"))
    else:
        items.append(f"{prefix}={value}")
    return items


def argument_f1(pred_calls: List[Dict[str, Any]], ref_calls: List[Dict[str, Any]]) -> float:
    pred_items = []
    ref_items = []

    for call in pred_calls:
        pred_items.extend([f"{call['name']}:{x}" for x in flatten_args(call.get("arguments", {}))])
    for call in ref_calls:
        ref_items.extend([f"{call['name']}:{x}" for x in flatten_args(call.get("arguments", {}))])

    pred_set = set(pred_items)
    ref_set = set(ref_items)

    if not pred_set and not ref_set:
        return 1.0
    if not pred_set or not ref_set:
        return 0.0

    tp = len(pred_set & ref_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(ref_set) if ref_set else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def extract_available_tool_names(tools_text: str) -> set:
    parsed = parse_json_object(tools_text)
    names = set()

    def walk(obj: Any):
        if isinstance(obj, dict):
            possible = obj.get("name") or obj.get("function", {}).get("name") if isinstance(obj.get("function"), dict) else obj.get("name")
            if possible:
                names.add(str(possible))
            for value in obj.values():
                walk(value)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(parsed)
    return names


def score_prediction(prediction: str, reference: str, tools_text: str) -> Dict[str, Any]:
    pred_calls, pred_valid = parse_tool_calls(prediction)
    ref_calls, ref_valid = parse_tool_calls(reference)

    pred_names = [call["name"] for call in pred_calls]
    ref_names = [call["name"] for call in ref_calls]
    available_names = extract_available_tool_names(tools_text)

    no_call_match = not pred_calls and not ref_calls
    function_correct = pred_names == ref_names
    hallucinated = bool(available_names) and any(name not in available_names for name in pred_names)

    return {
        "format_valid": bool(pred_valid or no_call_match),
        "function_correct": bool(function_correct),
        "argument_f1": argument_f1(pred_calls, ref_calls),
        "hallucinated": bool(hallucinated),
        "pred_calls": pred_calls,
        "ref_calls": ref_calls,
    }

## 9. Load the Base Model in 4-bit

This loads the base model for both baseline evaluation and QLoRA training. The model is quantized to 4-bit NF4 to fit on free Kaggle GPUs.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

print("Base model loaded.")

## 10. Baseline Evaluation on the Base Model


In [ ]:
def generate_response(model, prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model(model, dataset: Dataset, label: str, limit: Optional[int] = None) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []
    eval_count = len(dataset) if limit is None else min(limit, len(dataset))

    for idx in range(eval_count):
        sample = dataset[idx]
        prediction = generate_response(model, sample["prompt"])
        scores = score_prediction(prediction, sample["assistant"], sample.get("tools", "[]"))
        rows.append({
            "model": label,
            "index": idx,
            "prompt": sample["prompt"],
            "reference": sample["assistant"],
            "prediction": prediction,
            "format_valid": scores["format_valid"],
            "function_correct": scores["function_correct"],
            "argument_f1": scores["argument_f1"],
            "hallucinated": scores["hallucinated"],
            "pred_calls": json.dumps(scores["pred_calls"], ensure_ascii=False),
            "ref_calls": json.dumps(scores["ref_calls"], ensure_ascii=False),
        })

    df = pd.DataFrame(rows)
    summary = {
        "model": label,
        "n": len(df),
        "format_accuracy": float(df["format_valid"].mean()),
        "function_accuracy": float(df["function_correct"].mean()),
        "argument_f1": float(df["argument_f1"].mean()),
        "hallucination_rate": float(df["hallucinated"].mean()),
    }
    return df, summary


BASELINE_EVAL_LIMIT = TEST_SAMPLES
base_eval_df, base_summary = evaluate_model(base_model, test_ds, "base", limit=BASELINE_EVAL_LIMIT)

base_eval_df.to_csv(RESULTS_DIR / "base_predictions.csv", index=False)
print(pd.DataFrame([base_summary]))

## 11. Configure QLoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 12. Train with TRL SFTTrainer

In [ ]:
train_sft_ds = train_ds.remove_columns(
    [col for col in train_ds.column_names if col != "text"]
)

training_args = TrainingArguments(
    output_dir=str(PROJECT_DIR / "checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=50,
    logging_steps=25,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_sft_ds,
    args=training_args,
)

trainer.train()

## 13. Save the LoRA Adapter

In [ ]:
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved adapter to:", ADAPTER_DIR)